# Empirical Results: Adaptive Boolean Oracle (Marena 2026)
**Catholic University of America**

This notebook generates all empirical figures for the paper:
1. Adaptive vs Uniform error vs sample count (main result)
2. Tightness sweep over (N, K) pairs
3. NISQ noise crossover
4. k-Forrelation quantum vs classical separation
5. Kernel vs linear shadow accuracy

Runtime on Colab CPU: ~15 minutes total.

## Cell 1 — Install and clone

In [ ]:
import os

# Clone repo and install
if not os.path.exists('quantum_oracle_sketching'):
    !git clone https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git

%cd quantum_oracle_sketching
!pip install -q -e ".[dev,noise,kernel]"

import sys
sys.path.insert(0, 'src')

# Verify imports
from qos.theory.adaptive_lower_bound import (
    compute_bounds, tightness_sweep,
    uniform_vs_adaptive_error_comparison,
    adversarial_sparse_function,
    nisq_adaptive_crossover,
)
from qos.core.oracle_sketch import q_oracle_sketch_boolean, q_oracle_sketch_boolean_adaptive
from qos.primitives.noise_model import DepolarizingChannel, crossover_sample_count
from qos.data.generation import k_forrelation_data
import jax, jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight'
})
import os; os.makedirs('results', exist_ok=True)
print('Setup complete.')

## Cell 2 — Figure 1: Adaptive vs Uniform Error vs Sample Count
**This is the main result figure.** Shows adaptive achieves lower error at same M on adversarial sparse functions.
Expected runtime: ~3 minutes.

In [ ]:
N, K = 1024, 16  # sparsity ratio 1.6% — large enough contrast
sample_counts = [200, 500, 1000, 2000, 5000, 10000, 20000]

print(f'Running N={N}, K={K}, {len(sample_counts)} sample counts, 20 trials each...')
results = uniform_vs_adaptive_error_comparison(
    N=N, K=K,
    sample_counts=sample_counts,
    num_trials=20,
    key=jax.random.PRNGKey(0),
)

df1 = pd.DataFrame(results)
df1.to_csv('results/fig1_adaptive_vs_uniform.csv', index=False)
print(df1.to_string(index=False))

# Theoretical bound lines
M_arr = np.array(sample_counts)
# From Zhao Thm D.12: error ~ sqrt(N pi^2 / M)
theory_uniform = np.sqrt(N * np.pi**2 / M_arr)
# Support-restricted adaptive: error ~ sqrt(K pi^2 / M)
theory_adaptive = np.sqrt(K * np.pi**2 / M_arr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(df1['M'], df1['uniform_error'], yerr=df1['uniform_std'],
            fmt='o-', color='#d62728', label=f'Uniform QOS (N={N})', capsize=4)
ax.errorbar(df1['M'], df1['adaptive_error'], yerr=df1['adaptive_std'],
            fmt='s-', color='#1f77b4', label=f'Adaptive QOS (K={K})', capsize=4)
ax.plot(M_arr, theory_uniform, 'r--', alpha=0.5, label=r'Theory: $\sqrt{N\pi^2/M}$')
ax.plot(M_arr, theory_adaptive, 'b--', alpha=0.5, label=r'Theory: $\sqrt{K\pi^2/M}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Sample count M')
ax.set_ylabel(r'$L_\infty$ error on $\mathrm{supp}(f)$')
ax.set_title(f'Adaptive vs Uniform Oracle Sketch Error\n(N={N}, K={K}, sparsity={K/N:.1%})')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.savefig('results/fig1_adaptive_vs_uniform.pdf')
plt.savefig('results/fig1_adaptive_vs_uniform.png')
plt.show()
print(f'N/K improvement factor: {N/K:.0f}x')

## Cell 3 — Figure 2: Tightness Sweep over (N, K)
Verifies the lower bound holds across dimensions and sparsity levels.
Expected runtime: ~1 minute.

In [ ]:
N_values = [128, 256, 512, 1024]
sparsity_ratios = [0.01, 0.05, 0.1, 0.25]
epsilon = 0.1

sweep = tightness_sweep(N_values, sparsity_ratios, epsilon=epsilon)

rows = []
for r in sweep:
    rows.append({
        'N': r.N, 'K': r.K,
        'sparsity': r.K/r.N,
        'M_lower': r.M_lower_bound,
        'M_adaptive': r.M_adaptive_upper,
        'M_uniform': r.M_uniform_upper,
        'improvement_factor': r.improvement_factor,
        'tight': r.is_tight,
    })
df2 = pd.DataFrame(rows)
df2.to_csv('results/fig2_tightness_sweep.csv', index=False)
print(df2.to_string(index=False))

# Heatmap of improvement factor N/K
pivot = df2.pivot(index='N', columns='sparsity', values='improvement_factor')
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(sparsity_ratios)))
ax.set_xticklabels([f'{r:.0%}' for r in sparsity_ratios])
ax.set_yticks(range(len(N_values)))
ax.set_yticklabels(N_values)
ax.set_xlabel('Sparsity K/N'); ax.set_ylabel('N')
ax.set_title('Improvement Factor N/K (Adaptive vs Uniform)')
for i in range(len(N_values)):
    for j in range(len(sparsity_ratios)):
        ax.text(j, i, f'{pivot.values[i,j]:.0f}x',
                ha='center', va='center', fontsize=10, color='black')
plt.colorbar(im, ax=ax, label='N/K')
plt.savefig('results/fig2_tightness_sweep.pdf')
plt.savefig('results/fig2_tightness_sweep.png')
plt.show()

## Cell 4 — Figure 3: NISQ Noise Crossover
Shows M* vs noise rate — the point where adaptive improvement is nullified by gate noise.
Expected runtime: ~30 seconds.

In [ ]:
N, K = 512, 16
noise_rates = np.logspace(-4, -1, 30)  # 0.01% to 10% per gate
circuit_depth = 50
epsilon_target = 0.3

rows = []
for p in noise_rates:
    r = nisq_adaptive_crossover(N, K, float(p), circuit_depth, epsilon_target)
    rows.append({
        'noise_rate': p,
        'epsilon_noise': r['epsilon_noise'],
        'epsilon_sketch_budget': r['epsilon_sketch_budget'],
        'M_adaptive': r['M_adaptive_crossover'],
        'M_uniform': r['M_uniform_crossover'],
        'beneficial': r['adaptive_still_beneficial'],
    })
df3 = pd.DataFrame(rows)
df3.to_csv('results/fig3_nisq_crossover.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: M* vs noise rate
ax1.loglog(df3['noise_rate'], df3['M_adaptive'], 'b-', label=f'Adaptive (K={K})')
ax1.loglog(df3['noise_rate'], df3['M_uniform'], 'r-', label=f'Uniform (N={N})')
ax1.axvline(x=epsilon_target/circuit_depth, color='gray', linestyle='--',
            label=r'$\varepsilon_{\rm noise} = \varepsilon_{\rm target}$')
ax1.set_xlabel('Per-gate noise rate p')
ax1.set_ylabel('Crossover sample count M*')
ax1.set_title('NISQ Crossover: M* vs Noise Rate')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Right: remaining sketch budget vs noise rate
ax2.semilogx(df3['noise_rate'], df3['epsilon_sketch_budget'], 'g-')
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax2.set_xlabel('Per-gate noise rate p')
ax2.set_ylabel(r'Remaining budget $\varepsilon_{\rm sketch}$')
ax2.set_title(f'Sketch Budget After Noise\n(depth={circuit_depth}, target={epsilon_target})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/fig3_nisq_crossover.pdf')
plt.savefig('results/fig3_nisq_crossover.png')
plt.show()

## Cell 5 — Figure 4: k-Forrelation Quantum vs Classical Separation
Shows estimated vs exact k-Forrelation value, and classical lower bound scaling with k.
Expected runtime: ~3 minutes.

In [ ]:
n_bits = 8  # N = 2^8 = 256
k_values = [2, 3, 4, 5]
num_trials = 10
epsilon = 0.1

rows = []
for k in k_values:
    errors, classical_bounds = [], []
    for trial in range(num_trials):
        key = jax.random.PRNGKey(trial * 100 + k)
        gen = k_forrelation_data(n=n_bits, k=k, key=key)
        funcs = gen.sample_functions(key)
        exact = gen.compute_exact_forrelation(funcs)
        # Build oracle diagonal from all k layers
        from qos.utils.numerical import unnormalized_hadamard_transform
        state = funcs[-1].astype(jnp.float32)
        for fi in reversed(funcs[:-1]):
            state = unnormalized_hadamard_transform(state) / gen.dim
            state = fi.astype(jnp.float32) * state
        truth = ((1.0 - jnp.sign(state)) / 2.0).astype(jnp.int32)
        from qos.core.oracle_sketch import q_oracle_sketch_boolean
        diag, _ = q_oracle_sketch_boolean(truth, 5000)
        est = gen.quantum_query_algorithm(diag)
        errors.append(abs(float(est) - float(exact)))
        classical_bounds.append(gen.classical_streaming_complexity(epsilon))
    rows.append({
        'k': k,
        'mean_error': float(np.mean(errors)),
        'std_error': float(np.std(errors)),
        'classical_bound': int(np.mean(classical_bounds)),
    })

df4 = pd.DataFrame(rows)
df4.to_csv('results/fig4_k_forrelation.csv', index=False)
print(df4.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.errorbar(df4['k'], df4['mean_error'], yerr=df4['std_error'],
             fmt='o-', color='purple', capsize=5)
ax1.set_xlabel('k'); ax1.set_ylabel('|estimated - exact|')
ax1.set_title('QOS Estimator Error vs k-Forrelation Order')
ax1.grid(True, alpha=0.3)

ax2.semilogy(df4['k'], df4['classical_bound'], 's-', color='red',
             label=r'Classical lower bound $\Omega(N^{1-1/k}/\varepsilon^2)$')
ax2.set_xlabel('k'); ax2.set_ylabel('Sample complexity lower bound')
ax2.set_title('Classical Streaming Lower Bound vs k')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/fig4_k_forrelation.pdf')
plt.savefig('results/fig4_k_forrelation.png')
plt.show()

## Cell 6 — Figure 5: Kernel vs Linear Shadow Accuracy
Compares interferometric kernel shadow against linear shadow on separable data.
Expected runtime: ~2 minutes.

In [ ]:
from qos.core.state_sketch import (
    q_state_sketch_flat, fit_kernel_svm_from_states,
    q_interferometric_kernel_shadow,
)

def run_shadow_comparison(dim, n_train, n_test, sketch_samples, key):
    k1, k2, k3 = jax.random.split(key, 3)
    # Linearly separable data in R^dim
    X_train = jax.random.normal(k1, (n_train, dim))
    y_train = jnp.sign(X_train[:, 0]).astype(jnp.int32)  # label by first feature
    X_test = jax.random.normal(k2, (n_test, dim))
    y_test = jnp.sign(X_test[:, 0]).astype(jnp.int32)

    # Sketch all vectors
    sketch_train = jnp.stack([q_state_sketch_flat(X_train[i], sketch_samples)[0]
                               for i in range(n_train)])
    sketch_test = jnp.stack([q_state_sketch_flat(X_test[i], sketch_samples)[0]
                              for i in range(n_test)])

    # Fit kernel SVM on sketched states
    alpha = fit_kernel_svm_from_states(sketch_train, y_train)

    # Kernel shadow predictions
    kernel_preds = jnp.array([
        q_interferometric_kernel_shadow(sketch_train, y_train, alpha, sketch_test[i])
        for i in range(n_test)
    ])
    # Linear shadow: nearest-neighbor by inner product
    overlaps = jnp.abs(jnp.einsum('id,jd->ij', sketch_test, sketch_train))**2
    linear_preds = y_train[jnp.argmax(overlaps, axis=1)]

    kernel_acc = float(jnp.mean(kernel_preds == y_test))
    linear_acc = float(jnp.mean(linear_preds == y_test))
    return kernel_acc, linear_acc

dims = [16, 32, 64, 128]
rows = []
for d in dims:
    k_accs, l_accs = [], []
    for trial in range(8):
        ka, la = run_shadow_comparison(
            dim=d, n_train=20, n_test=40,
            sketch_samples=2000,
            key=jax.random.PRNGKey(trial * 7 + d)
        )
        k_accs.append(ka); l_accs.append(la)
    rows.append({
        'dim': d,
        'kernel_acc': np.mean(k_accs), 'kernel_std': np.std(k_accs),
        'linear_acc': np.mean(l_accs), 'linear_std': np.std(l_accs),
    })

df5 = pd.DataFrame(rows)
df5.to_csv('results/fig5_kernel_vs_linear.csv', index=False)
print(df5.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(df5['dim'], df5['kernel_acc'], yerr=df5['kernel_std'],
            fmt='s-', color='#1f77b4', label='Kernel shadow', capsize=4)
ax.errorbar(df5['dim'], df5['linear_acc'], yerr=df5['linear_std'],
            fmt='o-', color='#ff7f0e', label='Linear shadow (nearest-neighbor)', capsize=4)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Perfect accuracy')
ax.set_xlabel('State dimension d')
ax.set_ylabel('Classification accuracy')
ax.set_title('Kernel vs Linear Interferometric Shadow\n(linearly separable data, 20 train / 40 test)')
ax.set_ylim(0.4, 1.05); ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig('results/fig5_kernel_vs_linear.pdf')
plt.savefig('results/fig5_kernel_vs_linear.png')
plt.show()

## Cell 7 — Save all results summary

In [ ]:
print('=== All results saved to results/ ===')
for fname in sorted(os.listdir('results')):
    path = f'results/{fname}'
    size = os.path.getsize(path)
    print(f'  {fname:45s} {size:>8,} bytes')

# Download all results as a zip
import shutil
shutil.make_archive('marena2026_results', 'zip', 'results')
print('\nDownload marena2026_results.zip from the Files panel on the left.')